# 06_deepeval_testcase

06_deepeval_testcase.py — DeepEval LLMTestCase + 단일 메트릭

DeepEval 의 모든 평가는 LLMTestCase 에서 시작.
Ragas 와 필드 이름이 다르다 (input / actual_output / retrieval_context / expected_output).

In [1]:
import os, sys, ssl, certifi
# Windows 인증서 저장소 손상 우회(임베딩/HTTPS 로드 SSL 에러 방지)
ssl.SSLContext.load_default_certs = lambda self, *a, **k: self.load_verify_locations(certifi.where())
# 노트북 커널엔 __file__ 이 없으므로 스크립트 호환 위해 정의 + supp/ 를 import 경로에 추가
__file__ = os.path.join(os.getcwd(), '06_deepeval_testcase.py')
sys.path.insert(0, os.path.abspath('..'))

In [2]:
"""
06_deepeval_testcase.py — DeepEval LLMTestCase + 단일 메트릭

DeepEval 의 모든 평가는 LLMTestCase 에서 시작.
Ragas 와 필드 이름이 다르다 (input / actual_output / retrieval_context / expected_output).
"""
import sys as _sys
from pathlib import Path as _Path
_sys.path.insert(0, str(_Path(__file__).resolve().parent.parent))

from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric

from _common import banner, llm_unavailable
from _judges import deepeval_judge


def main() -> None:
    banner("DeepEval LLMTestCase — 필드 매핑 + Faithfulness")
    judge = deepeval_judge()
    if judge is None:
        llm_unavailable()
        return

    test_case = LLMTestCase(
        input="에펠탑은 어디에 있나?",
        actual_output="에펠탑은 프랑스 파리에 있습니다.",
        expected_output="에펠탑은 파리에 있습니다.",
        retrieval_context=["에펠탑은 프랑스 파리에 위치한다."],
    )

    print("\n  📋 LLMTestCase 4 필드 (Ragas 와 이름이 다름!):")
    print(f"    input              ← Ragas user_input")
    print(f"    actual_output      ← Ragas response")
    print(f"    expected_output    ← Ragas reference")
    print(f"    retrieval_context  ← Ragas retrieved_contexts")

    metric = FaithfulnessMetric(threshold=0.7, model=judge)
    try:
        metric.measure(test_case)
        print(f"\n  📊 Faithfulness")
        print(f"    score    = {metric.score}")
        print(f"    reason   = {metric.reason[:200] if metric.reason else '-'}")
        print(f"    passed   = {metric.is_successful()}")
    except Exception as e:
        print(f"\n  ⚠ {type(e).__name__}: {str(e)[:200]}")


if __name__ == "__main__":
    main()


📌 DeepEval LLMTestCase — 필드 매핑 + Faithfulness


D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')


  📋 LLMTestCase 4 필드 (Ragas 와 이름이 다름!):
    input              ← Ragas user_input
    actual_output      ← Ragas response
    expected_output    ← Ragas reference
    retrieval_context  ← Ragas retrieved_contexts



  📊 Faithfulness
    score    = 1.0
    reason   = The score is 1.00 because there are no contradictions between the actual output and the retrieval context, indicating perfect faithfulness.
    passed   = True
